# 새 질문과 Precision@5 확인
[프로젝트 추가] 이전 30문항과 다른 합성 질문 10개를 검색 전에 고정합니다. 사람의 교차 검토 전 Codex 내용 판정이므로 점수는 잠정값입니다. 기존 청킹·검색 규칙과 DB 검토 상태는 유지합니다.
아래 셀은 저장 결과를 읽기만 합니다. API를 호출하지 않습니다. 새로운 평가를 실행하려면 마지막 안내의 단계를 사용하세요.

In [ ]:
# [프로젝트 추가] 노트북 위치에 상관없이 프로젝트 폴더를 찾습니다.
from pathlib import Path
import sys
import json
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/car_search_rag/zzong_santafe_lag').is_dir())
if str(project / 'src') not in sys.path:
    sys.path.insert(0, str(project / 'src'))
from car_search_rag.zzong_santafe_lag.fresh_evaluation import current_plan
plan, folder = current_plan()
print('평가 질문 수:', len(plan['cases']))
print('질문 식별값:', plan['dataset_sha256'])
print('결과 폴더:', folder)


## 점수 뜻
- **Precision@5**: 상위 5개 후보 중 질문에 직접 답할 근거가 있는 후보의 비율. 2개면 2÷5=0.4입니다.
- **Hit@5**: 질문별로 상위 5개 중 직접 근거가 하나라도 있는지 확인한 비율.
- **MRR@5**: 첫 직접 근거의 순위. 1위는 1, 2위는 0.5입니다.
목차에 단어가 등장하거나 주제만 비슷하면 직접 근거로 세지 않습니다. 일부 요청에 직접 답하는 후보도 관련 후보로 세므로 답변 전체가 완전하다는 뜻은 아닙니다. 정답 ID만 비교한 이전 보조 Precision과 이번 내용 판정을 구분합니다.

In [ ]:
# [프로젝트 추가] 완료된 평가 점수를 읽습니다. 검색이나 생성 API를 다시 실행하지 않습니다.
metrics_path = folder / 'metrics.json'
if metrics_path.exists():
    report = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(json.dumps(report['metrics'], ensure_ascii=False, indent=2))
    print('사람의 교차 검토 완료:', report['human_confirmed'])
else:
    print('검색 후보 내용 판정과 점수 계산이 아직 완료되지 않았습니다.')


In [ ]:
# [프로젝트 추가] 사전 지정한 3문항의 저장 답변만 읽습니다. 발췌 전환을 생성 성공으로 세지 않습니다.
answer_path = folder / 'generation.json'
if answer_path.exists():
    generated = json.loads(answer_path.read_text(encoding='utf-8'))
    for row in generated['results']:
        saved_answer = row.get('answer', {})
        print('\n질문 번호:', row['case_id'])
        print('기대 내용:', row['required_facts'])
        print('답변 상태:', saved_answer.get('status', '실행 중'))
        print('검사 오류 종류:', saved_answer.get('generation_error_type', '없음'))
        print('사용 토큰:', saved_answer.get('token_usage'))
        print('저장 답변:', saved_answer.get('answer', '실행 중이거나 확인이 필요합니다.'))
else:
    print('생성 확인 결과가 아직 없습니다.')


In [ ]:
# [프로젝트 추가] 직접 근거 선택 문제와 아직 확정하지 못한 생성 오류를 구분해 읽습니다.
summary_path = project / 'src/car_search_rag/zzong_santafe_lag/fresh_evaluation_summary_20261005.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
for row in summary['generation_review']:
    print(row['case_id'], row['review'])
print('다음 제안:', summary['next_proposed_work'])
print('이번 평가에서 챗봇 답변 규칙을 수정했나요?:', summary['production_answer_rules_changed'])


## 단계별 실행
프로젝트 루트의 PowerShell에서 `$env:PYTHONPATH='src'`를 설정하고 `.\.venv\Scripts\python.exe -m car_search_rag.zzong_santafe_lag.fresh_evaluation` 뒤에 단계를 붙입니다.
1. `prepare`: 개인 DB를 읽고 질문·정답 기준을 고정. API 호출 없음.
2. `collect`: 두 모델 검색 및 원문 전문 보존. OpenAI 질문 임베딩 최대 10회. 성공 기록 재사용.
3. 후보 판정: `review_queue.json`의 각 원문을 읽고 `fresh_relevance_review_20261005.json`에 관련 여부·근거/이유를 기록한 뒤 `labels`로 식별값에 연결합니다. 미판정은 임의로 채점하지 않습니다. 사람 교차 검토 결과는 원래 결과를 보존한 별도 평가 버전에 반영해야 합니다.
4. `score`: 판정표로 점수 계산. API 호출 없음.
5. `generate`: 검색해 둔 근거로 미리 정한 3문항만 생성. 생성 API 최대 3회. 시작 기록이 남은 건은 자동 재실행하지 않습니다.
현재 도구는 이번 고정 평가용입니다. 새 질문 버전은 준비 코드를 먼저 검토합니다. 자세한 결과는 [쉬운 평가 안내](./fresh_evaluation_guide.md)에서 확인하세요. 이미 결과가 있으므로 지금은 위의 읽기 셀만 실행하면 됩니다.